# How Good Is Your Chat?
## Evaluating GenAI Applications for Recovery Detection

*WiDS Data Science Conference Workshop, AXA Switzerland*

---

In this notebook you will:

1. Set up a simple RAG pipeline over a synthetic insurance claim case
2. Generate answers to evaluation questions using an LLM
3. Build an answer correctness evaluator using LLM-as-a-judge
4. Build a faithfulness evaluator to detect hallucinations
5. Stress-test the judge to understand its failure modes

> **Time required:** ~30 minutes
> **What you need:** An [OpenRouter](https://openrouter.ai) API key (free tier is enough)


---
## Section 1: Setup

Run the two cells below. The first installs dependencies (output is suppressed). The second sets up the API client.

In [1]:
%%capture
!pip install openai pandas

In [9]:
import json
import re
import pandas as pd
from openai import OpenAI

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 120)

OPENROUTER_API_KEY = "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"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

MODEL = "deepseek/deepseek-v4-flash"
print(f"✓ Client configured. Using model: {MODEL}")

✓ Client configured. Using model: openai/gpt-4o-mini


In [4]:
def call_llm(prompt: str, system: str = None) -> str:
    """Call the LLM and return the text response."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
    )
    return response.choices[0].message.content


def call_llm_json(prompt: str, system: str = None) -> dict:
    """Call the LLM and return a parsed JSON dict."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


print("✓ Helper functions defined.")


✓ Helper functions defined.


---
## Section 2: The Scenario — A Synthetic Recovery Case

We have a fictional car accident claim with three documents:

| Document | Contents |
|---|---|
| **Police Report** | Date, location, parties, accident description, witness statements |
| **Medical Report** | Injuries, treatment, prognosis, inability to work |
| **Claim Form** | Damages claimed, amounts, supporting documents |

Read through the documents. Pay attention to the details, some inconsistencies are hiding in there.


In [5]:
POLICE_REPORT = """
POLICE REPORT
Report No.: ZH-2024-08-3421
Date of Incident: August 14, 2024
Location: Intersection of Bahnhofstrasse and Rosenweg, 8001 Zürich
Investigating Officer: Ltn. R. Frei, Kantonspolizei Zürich

PARTIES INVOLVED
Vehicle A: Anna Huber (DOB: 12.03.1982), resident of Winterthur.
  Vehicle: VW Golf, licence plate ZH 445 231.
Vehicle B: Thomas Müller (DOB: 07.11.1975), resident of Zürich.
  Vehicle: BMW 320i, licence plate ZH 112 087.

DESCRIPTION OF INCIDENT
At approximately 09:15, Vehicle A (Huber) was travelling south along Bahnhofstrasse
with right of way when Vehicle B (Müller) entered the intersection from Rosenweg,
failing to stop at the red traffic light. Vehicle B collided with the front-right
section of Vehicle A. The collision caused Vehicle A to be pushed sideways and come
to rest against the kerb.

VEHICLE DAMAGE ASSESSMENT
Vehicle A sustained minor to moderate damage to the front-right bumper and wheel arch.
Vehicle B sustained minor damage to the front bumper.

WITNESS STATEMENT
Ms. Sarah Keller (DOB: 04.06.1990), a pedestrian present at the scene, confirmed that
the traffic light facing Rosenweg was clearly red at the time of the collision. Her
statement is consistent with on-board camera footage retrieved from Vehicle A.

PRELIMINARY ASSESSMENT
Based on the evidence collected, Driver B (Thomas Müller) is considered at fault for
failing to comply with the traffic signal.

Filed by: Ltn. R. Frei | Date: August 14, 2024
"""

MEDICAL_REPORT = """
MEDICAL REPORT
Patient: Anna Huber
Date of Birth: 12.03.1982
Treating Physician: Dr. M. Baumann, Stadtspital Zürich Triemli
Date of Examination: August 14, 2024

REASON FOR PRESENTATION
Patient presented to the emergency department following a road traffic accident.
She reports sudden onset of neck pain and inability to move the left wrist without pain.

DIAGNOSES
1. Cervical whiplash injury (Grade II, Quebec Task Force classification)
2. Fracture of the distal radius, left wrist (confirmed by X-ray)

TREATMENT
- Cervical collar fitted for a period of 4 weeks
- Left wrist immobilised in a cast for 6 weeks
- Pain management: ibuprofen 400 mg, 3× daily
- Follow-up physiotherapy recommended after cast removal

PROGNOSIS
Full recovery expected within 3–4 months. No permanent impairment anticipated.

INABILITY TO WORK
The patient is employed as a graphic designer. Given the left wrist fracture and
associated pain, she is deemed fully unfit for work for a period of 6 weeks from the
date of the accident (August 14 to September 25, 2024). Partial resumption of work
may be possible from week 7 onward, subject to follow-up review.

Signed: Dr. M. Baumann | Date: August 14, 2024
"""

CLAIM_FORM = """
INSURANCE CLAIM FORM
Claim Reference: AXA-CH-2024-09-7821
Claimant: Anna Huber
Date of Submission: September 5, 2024
Insurer: AXA Switzerland | Policy Number: CH-882341-A

DESCRIPTION OF INCIDENT
On August 14, 2024, I was involved in a road traffic accident at the intersection of
Bahnhofstrasse and Rosenweg in Zürich. The other driver, Thomas Müller, ran a red
light and collided with my vehicle. I suffered physical injuries and my vehicle was
damaged as a result.

DAMAGES CLAIMED

1. Vehicle Repair Costs: CHF 15,000
   The front-right section of my VW Golf required extensive bodywork and complete
   replacement of the wheel arch assembly, bumper, headlight unit, and front frame
   alignment. Repair was carried out at AutoService AG Zürich (invoice attached).

2. Medical Expenses: CHF 8,000
   Costs incurred for emergency treatment, X-rays, cast fitting, medications, and
   two physiotherapy sessions at Stadtspital Zürich Triemli.

3. Lost Wages: CHF 22,000
   I was unable to work for 6 weeks. My gross monthly salary is CHF 8,800.
   Additionally, I lost a confirmed freelance contract (CHF 8,800) that could not
   be fulfilled during my recovery period.

TOTAL CLAIMED AMOUNT: CHF 45,000

SUPPORTING DOCUMENTS ATTACHED
- Vehicle repair invoice (AutoService AG Zürich)
- Medical receipts (Stadtspital Zürich Triemli)
- Employer confirmation of salary and absence period
- Freelance contract and client cancellation notice

Signature: Anna Huber | Date: September 5, 2024
"""

DOCUMENTS = {
    "Police Report": POLICE_REPORT,
    "Medical Report": MEDICAL_REPORT,
    "Claim Form": CLAIM_FORM,
}

print("✓ Documents loaded.")
print(f"  Police Report:  {len(POLICE_REPORT.split()):>4} words")
print(f"  Medical Report: {len(MEDICAL_REPORT.split()):>4} words")
print(f"  Claim Form:     {len(CLAIM_FORM.split()):>4} words")


✓ Documents loaded.
  Police Report:   223 words
  Medical Report:  190 words
  Claim Form:      219 words


In [6]:
EVAL_SET = [
    {
        "question": "Who was at fault in the accident?",
        "reference_answer": (
            "Based on the police report, Driver B (Thomas Müller) is at fault. "
            "He failed to stop at a red traffic light at the intersection of "
            "Bahnhofstrasse and Rosenweg, causing a collision with Anna Huber's vehicle."
        ),
    },
    {
        "question": "How long was the claimant unable to work?",
        "reference_answer": (
            "The medical report states that Anna Huber was fully unfit for work for "
            "6 weeks, from August 14 to September 25, 2024."
        ),
    },
    {
        "question": "What is the total claimed amount?",
        "reference_answer": (
            "The claim form lists a total of CHF 45,000 in damages, comprising "
            "CHF 15,000 for vehicle repairs, CHF 8,000 for medical expenses, and "
            "CHF 22,000 for lost wages."
        ),
    },
    {
        "question": "Were there any witnesses to the accident?",
        "reference_answer": (
            "Yes. Ms. Sarah Keller, a pedestrian at the scene, confirmed that the "
            "traffic light facing Rosenweg was clearly red at the time of the "
            "collision. Her statement is corroborated by on-board camera footage "
            "from Vehicle A."
        ),
    },
    {
        "question": "What injuries did the claimant sustain?",
        "reference_answer": (
            "Anna Huber sustained a cervical whiplash injury (Grade II) and a "
            "fracture of the distal radius of her left wrist, confirmed by X-ray."
        ),
    },
    {
        "question": ("Is there any inconsistency between the documents that might warrant further investigation?"),
        "reference_answer": (
            "Yes. The police report describes 'minor to moderate damage' to Vehicle A, "
            "while the claim form lists CHF 15,000 for vehicle repairs — a substantial "
            "sum for what the police characterised as minor damage. This discrepancy "
            "may warrant a review of the vehicle repair invoice."
        ),
    },
]

print(f"✓ Evaluation set loaded: {len(EVAL_SET)} questions")
for i, item in enumerate(EVAL_SET, 1):
    print(f"  Q{i}: {item['question'][:70]}...")


✓ Evaluation set loaded: 6 questions
  Q1: Who was at fault in the accident?...
  Q2: How long was the claimant unable to work?...
  Q3: What is the total claimed amount?...
  Q4: Were there any witnesses to the accident?...
  Q5: What injuries did the claimant sustain?...
  Q6: Is there any inconsistency between the documents that might warrant fu...


---
## Section 3: Generate Answers with a Simple RAG Pipeline

We simulate a RAG (Retrieval-Augmented Generation) system:

1. **Retrieve** the most relevant document(s) for each question using keyword overlap (no vector-based semantic search)
2. **Generate** an answer by sending the retrieved context + question to the LLM

We are only performing keyword search to keep this intentionally simple by focusing on *evaluation*, not the RAG architecture itself.


In [7]:
def retrieve_context(question: str, top_k: int = 2) -> list[tuple[str, str]]:
    """Return the top-k most relevant documents using keyword overlap scoring."""
    # Normalise: lowercase, strip punctuation, split into words
    q_words = set(re.sub(r"[^\w\s]", "", question.lower()).split())

    scores = {}
    for name, doc in DOCUMENTS.items():
        doc_words = set(re.sub(r"[^\w\s]", "", doc.lower()).split())
        scores[name] = len(q_words & doc_words)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    # Return documents with at least one matching word, up to top_k
    return [(name, DOCUMENTS[name]) for name, score in ranked[:top_k] if score > 0]


def generate_answer(question: str, context_docs: list[tuple[str, str]]) -> str:
    """Generate an answer given the question and retrieved context documents."""
    if not context_docs:
        return "No relevant documents found."

    context_str = "\n\n---\n\n".join(f"[{name}]\n{doc}" for name, doc in context_docs)
    prompt = (
        f"You are a helpful assistant for insurance recovery experts.\n"
        f"Answer the question based only on the provided documents.\n\n"
        f"DOCUMENTS:\n{context_str}\n\n"
        f"QUESTION: {question}\n\n"
        f"Answer concisely and factually. Cite the source document when relevant."
    )
    return call_llm(prompt)


print("✓ Retrieval and generation functions defined.")


✓ Retrieval and generation functions defined.


In [10]:
print("Generating answers... (this makes one API call per question)\n")

rows = []
for i, item in enumerate(EVAL_SET, 1):
    q = item["question"]
    ref = item["reference_answer"]

    context_docs = retrieve_context(q)
    retrieved_names = [name for name, _ in context_docs]
    retrieved_text = "\n\n---\n\n".join(f"[{n}]\n{d}" for n, d in context_docs)

    answer = generate_answer(q, context_docs)

    rows.append(
        {
            "q_id": i,
            "question": q,
            "reference_answer": ref,
            "retrieved_docs": ", ".join(retrieved_names),
            "generated_answer": answer,
        }
    )
    print(f"Q{i} done — retrieved: {retrieved_names}")

results_df = pd.DataFrame(rows)
print("\n✓ All answers generated.")


Generating answers... (this makes one API call per question)



AuthenticationError: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}

In [ ]:
# Display results — inspect each generated answer
for _, row in results_df.iterrows():
    print(f"{'=' * 70}")
    print(f"Q{row['q_id']}: {row['question']}")
    print(f"  Retrieved: {row['retrieved_docs']}")
    print(f"  Reference: {row['reference_answer']}")
    print(f"  Generated: {row['generated_answer']}")


Q1: Who was at fault in the accident?
  Retrieved: Claim Form, Police Report
  Reference: Based on the police report, Driver B (Thomas Müller) is at fault. He failed to stop at a red traffic light at the intersection of Bahnhofstrasse and Rosenweg, causing a collision with Anna Huber's vehicle.
  Generated: Thomas Müller was at fault in the accident for failing to stop at the red traffic light. This is confirmed in the police report, which states that "Driver B (Thomas Müller) is considered at fault for failing to comply with the traffic signal" (Police Report, ZH-2024-08-3421).
Q2: How long was the claimant unable to work?
  Retrieved: Claim Form, Police Report
  Reference: The medical report states that Anna Huber was fully unfit for work for 6 weeks, from August 14 to September 25, 2024.
  Generated: The claimant, Anna Huber, was unable to work for 6 weeks. This information is stated in the Claim Form under the "Lost Wages" section.
Q3: What is the total claimed amount?
  Retrieved:

In [ ]:
# ─── CHECKPOINT ──────────────────────────────────────────────────────────────
# Set USE_PRECOMPUTED = True to skip the API calls above and load pre-baked results.
USE_PRECOMPUTED = False

PRECOMPUTED_ANSWERS = [
    {
        "q_id": 1,
        "question": "Who was at fault in the accident?",
        "reference_answer": "Based on the police report, Driver B (Thomas Müller) is at fault. He failed to stop at a red traffic light at the intersection of Bahnhofstrasse and Rosenweg, causing a collision with Anna Huber's vehicle.",
        "retrieved_docs": "Claim Form, Police Report",
        "generated_answer": "Thomas Müller was at fault in the accident for failing to comply with the traffic signal. This is confirmed in the police report, which states that he entered the intersection from Rosenweg, ran a red light, and collided with Anna Huber's vehicle (Police Report, ZH-2024-08-3421).",
    },
    {
        "q_id": 2,
        "question": "How long was the claimant unable to work?",
        "reference_answer": "The medical report states that Anna Huber was fully unfit for work for 6 weeks, from August 14 to September 25, 2024.",
        "retrieved_docs": "Claim Form, Police Report",
        "generated_answer": "The claimant, Anna Huber, was unable to work for 6 weeks. This information is stated in the Claim Form under the 'Lost Wages' section.",
    },
    {
        "q_id": 3,
        "question": "What is the total claimed amount?",
        "reference_answer": "The claim form lists a total of CHF 45,000 in damages, comprising CHF 15,000 for vehicle repairs, CHF 8,000 for medical expenses, and CHF 22,000 for lost wages.",
        "retrieved_docs": "Claim Form, Police Report",
        "generated_answer": "The total claimed amount is CHF 45,000, as stated in the Insurance Claim Form submitted by Anna Huber.",
    },
    {
        "q_id": 4,
        "question": "Were there any witnesses to the accident?",
        "reference_answer": "Yes. Ms. Sarah Keller, a pedestrian at the scene, confirmed that the traffic light facing Rosenweg was clearly red at the time of the collision. Her statement is corroborated by on-board camera footage from Vehicle A.",
        "retrieved_docs": "Medical Report, Claim Form",
        "generated_answer": "The provided documents do not mention any witnesses to the accident. The information is solely based on Anna Huber's account of the incident in the Claim Form.",
    },
    {
        "q_id": 5,
        "question": "What injuries did the claimant sustain?",
        "reference_answer": "Anna Huber sustained a cervical whiplash injury (Grade II) and a fracture of the distal radius of her left wrist, confirmed by X-ray.",
        "retrieved_docs": "Claim Form, Police Report",
        "generated_answer": "The claimant, Anna Huber, suffered physical injuries as a result of the road traffic accident. The specific details of the injuries are not provided in the documents, but they are mentioned in the claim form under 'Medical Expenses', which includes costs for emergency treatment, X-rays, cast fitting, medications, and physiotherapy sessions (Claim Form).",
    },
    {
        "q_id": 6,
        "question": "Is there any inconsistency between the documents that might warrant further investigation?",
        "reference_answer": "Yes. The police report describes 'minor to moderate damage' to Vehicle A, while the claim form lists CHF 15,000 for vehicle repairs — a substantial sum for what the police characterised as minor damage. This discrepancy may warrant a review of the vehicle repair invoice.",
        "retrieved_docs": "Claim Form, Police Report",
        "generated_answer": "Yes, there is an inconsistency between the documents regarding the assessment of vehicle damage. In the Claim Form submitted by Anna Huber, she claims vehicle repair costs of CHF 15,000 for extensive damage to her VW Golf, including complete replacement of the wheel arch assembly, bumper, headlight unit, and front frame alignment. However, the Police Report indicates that Vehicle A (Huber) sustained 'minor to moderate damage' to the front-right bumper and wheel arch (Police Report, ZH-2024-08-3421). This discrepancy between the claimed repair costs and the police assessment of damage may warrant further investigation.",
    },
]

if USE_PRECOMPUTED:
    results_df = pd.DataFrame(PRECOMPUTED_ANSWERS)
    print("✓ Loaded pre-computed answers.")
else:
    print("ℹ Using live-generated answers (set USE_PRECOMPUTED = True to use cached results).")


✓ Loaded pre-computed answers.


### 💬 Discussion

Before we automate the evaluation — take a moment to read the generated answers.

- Which ones look correct? Which seem off?
- Would you trust all of them equally?
- Notice Q6: does the generated answer add anything beyond what's in the documents?

*Discuss with a neighbour for 1–2 minutes.*


---
## Section 4: Build an Answer Correctness Evaluator

We'll use an LLM as a judge. The judge receives three inputs:
- The **question**
- The **reference answer** (our ground truth)
- The **generated answer** (what the RAG system produced)

It scores the generated answer on a scale of **1–5** and explains its reasoning.

### The Judge Prompt

```
SYSTEM:
You are an expert evaluator for an insurance claim Q&A system.
Your task is to assess whether a generated answer correctly addresses
the question compared to a reference answer.

USER:
Question: {question}
Reference Answer: {reference_answer}
Generated Answer: {generated_answer}

Score the generated answer from 1 to 5:
  5 – Fully correct. All key facts match the reference.
  4 – Mostly correct. Minor omissions or phrasing differences.
  3 – Partially correct. Some key facts are right, some are missing or wrong.
  2 – Mostly incorrect. A few correct elements but major errors.
  1 – Completely wrong or unrelated.

Return a JSON object with keys "score" (integer) and "reasoning" (string).
```

The full prompt is shown so you can see exactly what the judge "sees". The quality of the judge is only as good as the quality of the prompt.


In [ ]:
CORRECTNESS_SYSTEM = (
    "You are an expert evaluator for an insurance claim Q&A system. "
    "Your task is to assess whether a generated answer correctly addresses "
    "the question compared to a reference answer."
)

CORRECTNESS_PROMPT_TEMPLATE = """\
Question: {question}

Reference Answer:
{reference_answer}

Generated Answer:
{generated_answer}

Score the generated answer from 1 to 5:
  5 - Fully correct. All key facts match the reference.
  4 - Mostly correct. Minor omissions or phrasing differences, nothing misleading.
  3 - Partially correct. Some key facts right, but important facts missing or wrong.
  2 - Mostly incorrect. A few correct elements but major errors present.
  1 - Completely wrong, hallucinated, or unrelated to the question.

Return a JSON object with exactly two keys:
  "score": integer (1-5)
  "reasoning": string (1-2 sentences explaining your score)
"""


def evaluate_correctness(question: str, reference: str, generated: str) -> dict:
    """Judge the correctness of a generated answer against a reference. Returns {score, reasoning}."""
    prompt = CORRECTNESS_PROMPT_TEMPLATE.format(
        question=question,
        reference_answer=reference,
        generated_answer=generated,
    )
    result = call_llm_json(prompt, system=CORRECTNESS_SYSTEM)
    return {"score": int(result["score"]), "reasoning": result["reasoning"]}


print("✓ Correctness evaluator defined.")


✓ Correctness evaluator defined.


In [ ]:
print("Running correctness evaluator... (one API call per question)\n")

correctness_rows = []
for _, row in results_df.iterrows():
    result = evaluate_correctness(row["question"], row["reference_answer"], row["generated_answer"])
    correctness_rows.append(
        {
            "q_id": row["q_id"],
            "question": row["question"],
            "generated_answer": row["generated_answer"],
            "correctness_score": result["score"],
            "correctness_reasoning": result["reasoning"],
        }
    )
    print(f"Q{row['q_id']} → score {result['score']}/5")

correctness_df = pd.DataFrame(correctness_rows)
print("\n✓ Correctness evaluation complete.")


Running correctness evaluator... (one API call per question)

Q1 → score 5/5
Q2 → score 5/5
Q3 → score 5/5
Q4 → score 1/5
Q5 → score 2/5
Q6 → score 5/5

✓ Correctness evaluation complete.


In [ ]:
# Display the correctness results
for _, row in correctness_df.iterrows():
    print(f"{'=' * 70}")
    print(f"Q{row['q_id']}: {row['question']}")
    print(f"  Answer:    {row['generated_answer'][:120]}...")
    print(f"  Score:     {row['correctness_score']}/5")
    print(f"  Reasoning: {row['correctness_reasoning']}")
    print()


Q1: Who was at fault in the accident?
  Answer:    Thomas Müller was at fault in the accident for failing to comply with the traffic signal. This is confirmed in the polic...
  Score:     5/5
  Reasoning: The generated answer accurately identifies Thomas Müller as at fault for running a red light, which aligns with the reference answer. It includes all key facts and provides additional detail from the police report, confirming the circumstances of the accident.

Q2: How long was the claimant unable to work?
  Answer:    The claimant, Anna Huber, was unable to work for 6 weeks. This information is stated in the Claim Form under the 'Lost W...
  Score:     5/5
  Reasoning: The generated answer accurately states that Anna Huber was unable to work for 6 weeks, which matches the reference answer. It also correctly identifies the source of the information as the Claim Form, aligning with the details provided in the reference.

Q3: What is the total claimed amount?
  Answer:    The total clai

In [ ]:
# ─── CHECKPOINT ──────────────────────────────────────────────────────────────
USE_PRECOMPUTED_CORRECTNESS = False

PRECOMPUTED_CORRECTNESS = [
    {
        "q_id": 1,
        "question": EVAL_SET[0]["question"],
        "generated_answer": PRECOMPUTED_ANSWERS[0]["generated_answer"],
        "correctness_score": 5,
        "correctness_reasoning": "The generated answer accurately identifies Thomas Müller as at fault for running a red light, which aligns with the reference answer. It includes all key facts and provides additional detail from the police report, confirming the circumstances of the accident.",
    },
    {
        "q_id": 2,
        "question": EVAL_SET[1]["question"],
        "generated_answer": PRECOMPUTED_ANSWERS[1]["generated_answer"],
        "correctness_score": 5,
        "correctness_reasoning": "The generated answer accurately states that Anna Huber was unable to work for 6 weeks, which matches the reference answer. It also correctly identifies the source of the information as the Claim Form, aligning with the details provided in the reference.",
    },
    {
        "q_id": 3,
        "question": EVAL_SET[2]["question"],
        "generated_answer": PRECOMPUTED_ANSWERS[2]["generated_answer"],
        "correctness_score": 5,
        "correctness_reasoning": "The generated answer correctly states the total claimed amount of CHF 45,000, which matches the reference answer. It also accurately attributes the claim to the Insurance Claim Form submitted by Anna Huber, providing a complete and correct response to the question.",
    },
    {
        "q_id": 4,
        "question": EVAL_SET[3]["question"],
        "generated_answer": PRECOMPUTED_ANSWERS[3]["generated_answer"],
        "correctness_score": 1,
        "correctness_reasoning": "The generated answer is completely wrong as it states there were no witnesses, while the reference answer clearly identifies a witness and provides corroborating evidence. This indicates a significant misunderstanding of the question's context.",
    },
    {
        "q_id": 5,
        "question": EVAL_SET[4]["question"],
        "generated_answer": PRECOMPUTED_ANSWERS[4]["generated_answer"],
        "correctness_score": 2,
        "correctness_reasoning": "The generated answer fails to specify the actual injuries sustained by the claimant, which are critical to the question. While it mentions related medical expenses, it does not provide the specific injuries of cervical whiplash and wrist fracture as stated in the reference answer.",
    },
    {
        "q_id": 6,
        "question": EVAL_SET[5]["question"],
        "generated_answer": PRECOMPUTED_ANSWERS[5]["generated_answer"],
        "correctness_score": 5,
        "correctness_reasoning": "The generated answer accurately identifies the inconsistency between the claimed repair costs and the police report's assessment of damage, mirroring the key facts presented in the reference answer. It provides specific details about the claim and the police report, making it fully correct.",
    },
]

if USE_PRECOMPUTED_CORRECTNESS:
    correctness_df = pd.DataFrame(PRECOMPUTED_CORRECTNESS)
    print("✓ Loaded pre-computed correctness scores.")
else:
    print("ℹ Using live correctness scores.")


ℹ Using live correctness scores.


### 💬 Discussion

- Do you agree with the judge's scores and reasoning?
- Q6 is interesting: the generated answer adds a recommendation ("independent damage assessment") not in the reference. The judge still scores it 5. Is that right?
- What does a correctness score of 5 tell you about faithfulness?


---
## Section 5: Add a Faithfulness Evaluator

Correctness measures whether the answer matches the reference. But **faithfulness** asks a different question:

> *Does every claim in the generated answer actually come from the retrieved documents?*

A faithful answer contains **no information beyond what was retrieved**: no hallucinations, no assumptions, no general knowledge slipped in. Notice that:

- An answer can be **correct but unfaithful**. For example, if it happens to be right despite adding information not in the context.
- An answer can be **faithful but incorrect**. For example, if the retrieved context is incomplete or wrong.

This evaluator doesn't need a reference answer. It only looks at the generated answer vs. the retrieved context.


In [ ]:
FAITHFULNESS_SYSTEM = (
    "You are an expert evaluator for an insurance claim Q&A system. "
    "Your task is to assess whether a generated answer is fully grounded in "
    "the provided source documents, with no unsupported claims."
)

FAITHFULNESS_PROMPT_TEMPLATE = """\
Question: {question}

Retrieved Context (the only information the system had access to):
{retrieved_context}

Generated Answer:
{generated_answer}

Evaluate whether every claim in the generated answer is supported by the retrieved context.

Verdict options:
  "faithful"           - Every claim can be traced directly to the context.
  "partially faithful" - Most claims are supported, but 1-2 minor additions or
                         inferences go beyond what the context states.
  "unfaithful"         - The answer contains significant claims not found in
                         the context, or directly contradicts the context.

Return a JSON object with exactly three keys:
  "verdict": one of "faithful", "partially faithful", "unfaithful"
  "unsupported_claims": list of strings (claims in the answer NOT found in context; empty list if faithful)
  "reasoning": string (1-2 sentences)
"""


def evaluate_faithfulness(question: str, context: str, generated: str) -> dict:
    """Judge whether a generated answer is faithful to the retrieved context."""
    prompt = FAITHFULNESS_PROMPT_TEMPLATE.format(
        question=question,
        retrieved_context=context,
        generated_answer=generated,
    )
    result = call_llm_json(prompt, system=FAITHFULNESS_SYSTEM)
    return {
        "verdict": result.get("verdict", "unknown"),
        "unsupported_claims": result.get("unsupported_claims", []),
        "reasoning": result.get("reasoning", ""),
    }


print("✓ Faithfulness evaluator defined.")


✓ Faithfulness evaluator defined.


In [ ]:
print("Running faithfulness evaluator... (one API call per question)\n")

faithfulness_rows = []
for _, row in results_df.iterrows():
    # Reconstruct the full retrieved context text
    retrieved_doc_names = [n.strip() for n in row["retrieved_docs"].split(",")]
    context_text = "\n\n---\n\n".join(
        f"[{name}]\n{DOCUMENTS[name]}" for name in retrieved_doc_names if name in DOCUMENTS
    )

    result = evaluate_faithfulness(row["question"], context_text, row["generated_answer"])
    faithfulness_rows.append(
        {
            "q_id": row["q_id"],
            "verdict": result["verdict"],
            "unsupported_claims": result["unsupported_claims"],
            "faithfulness_reasoning": result["reasoning"],
        }
    )
    unsupported = result["unsupported_claims"]
    print(f"Q{row['q_id']} → {result['verdict']}" + (f" | unsupported: {unsupported}" if unsupported else ""))

faithfulness_df = pd.DataFrame(faithfulness_rows)
print("\n✓ Faithfulness evaluation complete.")


Running faithfulness evaluator... (one API call per question)

Q1 → faithful
Q2 → faithful
Q3 → faithful
Q4 → faithful
Q5 → partially faithful | unsupported: ['The specific details of the injuries are not provided in the documents.']
Q6 → faithful

✓ Faithfulness evaluation complete.


In [ ]:
# Merge correctness and faithfulness into one view
combined_df = correctness_df.merge(faithfulness_df, on="q_id")

print(f"{'Q':<4} {'Score':>6} {'Verdict':<22} {'Unsupported Claims'}")
print("-" * 80)
for _, row in combined_df.iterrows():
    unsupported = "; ".join(row["unsupported_claims"]) if row["unsupported_claims"] else "—"
    print(f"Q{row['q_id']:<3} {row['correctness_score']:>5}/5  {row['verdict']:<22} {unsupported[:50]}")


Q     Score Verdict                Unsupported Claims
--------------------------------------------------------------------------------
Q1       5/5  faithful               —
Q2       5/5  faithful               —
Q3       5/5  faithful               —
Q4       1/5  faithful               —
Q5       2/5  partially faithful     The specific details of the injuries are not provi
Q6       5/5  faithful               —


In [ ]:
# ─── CHECKPOINT ──────────────────────────────────────────────────────────────
USE_PRECOMPUTED_FAITHFULNESS = False

PRECOMPUTED_FAITHFULNESS = [
    {
        "q_id": 1,
        "verdict": "faithful",
        "unsupported_claims": [],
        "faithfulness_reasoning": "All claims in the answer (red light, intersection, witness, camera footage) are directly supported by the Police Report.",
    },
    {
        "q_id": 2,
        "verdict": "faithful",
        "unsupported_claims": [],
        "faithfulness_reasoning": "The 6-week period and dates are directly stated in the Medical Report. The note about week 7 is also explicitly mentioned.",
    },
    {
        "q_id": 3,
        "verdict": "faithful",
        "unsupported_claims": [],
        "faithfulness_reasoning": "All amounts and categories match exactly what is stated in the Claim Form.",
    },
    {
        "q_id": 4,
        "verdict": "faithful",
        "unsupported_claims": [],
        "faithfulness_reasoning": "Details about the witness are not stated in the retrieved documents.",
    },
    {
        "q_id": 5,
        "verdict": "faithful",
        "unsupported_claims": [],
        "faithfulness_reasoning": "The specific details of the injuries are not provided in the documents.",
    },
    {
        "q_id": 6,
        "verdict": "partially faithful",
        "unsupported_claims": ["recommendation for an independent damage assessment"],
        "faithfulness_reasoning": "The core discrepancy is correctly identified from the documents, but the suggestion to conduct an 'independent damage assessment' is an inference not explicitly found in any retrieved document.",
    },
]

if USE_PRECOMPUTED_FAITHFULNESS:
    faithfulness_df = pd.DataFrame(PRECOMPUTED_FAITHFULNESS)
    combined_df = pd.DataFrame(PRECOMPUTED_CORRECTNESS).merge(faithfulness_df, on="q_id")
    print("✓ Loaded pre-computed faithfulness verdicts.")
else:
    print("ℹ Using live faithfulness verdicts.")


ℹ Using live faithfulness verdicts.


### Insight: Correctness ≠ Faithfulness

Look at **Q6** (the inconsistency question):

- **Correctness score: 5/5**, the judge says the answer is correct
- **Faithfulness: partially faithful**, the answer adds a recommendation ("independent damage assessment") not found in any document

This is a real tension in RAG evaluation:
- From a *correctness* perspective, the recommendation is reasonable and arguably helpful
- From a *faithfulness* perspective, the answer introduces information beyond what the system retrieved

Which matters more depends on your use case. For a recovery expert tool, you might prefer **strict faithfulness** to prevent the system from making recommendations that go beyond the evidence.


---
## Section 6: Stress-Test the Judge

Now let's deliberately break things to understand where the judge fails.

We'll inject three carefully crafted answers and see how both evaluators respond.


### Experiment 1: The Hallucinated Answer

We inject an answer to Q1 that contains a plausible but fabricated detail: a specific time of day ("14:30") that appears nowhere in any document. (The police report says the accident happened at approximately 09:15, so this is also factually wrong.)

**Question:** Will the faithfulness evaluator catch it?


In [ ]:
HALLUCINATED_ANSWER = (
    "Based on the Police Report, Thomas Müller (Driver B) is at fault. "
    "He failed to stop at a red traffic light at the intersection of "
    "Bahnhofstrasse and Rosenweg. The accident occurred at 14:30. "
    "This is supported by witness testimony from Ms. Sarah Keller."
)

q1 = EVAL_SET[0]["question"]
ref1 = EVAL_SET[0]["reference_answer"]
context1 = POLICE_REPORT  # Police report is the relevant document for Q1

print("=== Experiment 1: Hallucinated Time ===\n")
print(f"Injected answer:\n  {HALLUCINATED_ANSWER}\n")

corr_result = evaluate_correctness(q1, ref1, HALLUCINATED_ANSWER)
faith_result = evaluate_faithfulness(q1, context1, HALLUCINATED_ANSWER)

print(f"Correctness:  {corr_result['score']}/5")
print(f"  Reasoning:  {corr_result['reasoning']}\n")
print(f"Faithfulness: {faith_result['verdict']}")
print(f"  Unsupported: {faith_result['unsupported_claims']}")
print(
    f"  Reasoning:  {faith_result['faithfulness_reasoning'] if 'faithfulness_reasoning' in faith_result else faith_result['reasoning']}"
)


=== Experiment 1: Hallucinated Time ===

Injected answer:
  Based on the Police Report, Thomas Müller (Driver B) is at fault. He failed to stop at a red traffic light at the intersection of Bahnhofstrasse and Rosenweg. The accident occurred at 14:30. This is supported by witness testimony from Ms. Sarah Keller.

Correctness:  4/5
  Reasoning:  The generated answer correctly identifies Thomas Müller as at fault and provides the reason for the fault, matching the reference answer. However, it includes additional details about the time of the accident and witness testimony, which are not present in the reference answer, but these do not detract from the core information.

Faithfulness: partially faithful
  Unsupported: ['The accident occurred at 14:30.']
  Reasoning:  The generated answer correctly identifies Thomas Müller as at fault and references the witness testimony, but it inaccurately states the time of the accident as 14:30, while the context specifies it occurred at approximately

**What to look for:**
- Does the faithfulness evaluator flag "14:30" as unsupported? It should, no document mentions this time.
- Does the correctness evaluator penalise the wrong time? It may score 4/5 or lower if it notices the factual error.
- This is a best-case scenario for the judge. The hallucination is a clear, specific fact. What about vaguer hallucinations?


### Experiment 2: Verbose but Wrong

We inject a long, confident, well-structured answer to Q3 (total claimed amount) that states the wrong total: **CHF 50,000** instead of CHF 45,000.

**Hypothesis:** The judge may give a higher score than deserved because the answer *sounds* authoritative and detailed, this is **verbosity bias**.


In [ ]:
VERBOSE_WRONG_ANSWER = (
    "The total claimed amount in this insurance matter is CHF 50,000. "
    "This figure reflects a comprehensive assessment of all damages sustained by Ms. Huber: "
    "Vehicle A (the VW Golf) suffered significant structural damage requiring complete "
    "replacement of the front assembly, suspension components, and electrical systems, "
    "totalling CHF 18,000. Medical expenses amount to CHF 10,000, covering emergency "
    "treatment at Stadtspital Zürich Triemli, specialist consultations, cast fitting, "
    "medications, and physiotherapy. Lost wages and income account for CHF 22,000, "
    "reflecting Ms. Huber's 6-week absence from her graphic design position and a "
    "confirmed freelance contract that could not be fulfilled. All amounts are supported "
    "by invoices and employer documentation."
)

q3 = EVAL_SET[2]["question"]
ref3 = EVAL_SET[2]["reference_answer"]
context3 = CLAIM_FORM  # Claim form is most relevant for Q3

print("=== Experiment 2: Verbose but Wrong ===\n")
print(f"Injected answer:\n  {VERBOSE_WRONG_ANSWER[:200]}...\n")

corr_result = evaluate_correctness(q3, ref3, VERBOSE_WRONG_ANSWER)
faith_result = evaluate_faithfulness(q3, context3, VERBOSE_WRONG_ANSWER)

print(f"Correctness:  {corr_result['score']}/5")
print(f"  Reasoning:  {corr_result['reasoning']}\n")
print(f"Faithfulness: {faith_result['verdict']}")
print(f"  Unsupported: {faith_result['unsupported_claims']}")
print(f"  Reasoning:  {faith_result.get('faithfulness_reasoning', faith_result.get('reasoning', ''))}")


=== Experiment 2: Verbose but Wrong ===

Injected answer:
  The total claimed amount in this insurance matter is CHF 50,000. This figure reflects a comprehensive assessment of all damages sustained by Ms. Huber: Vehicle A (the VW Golf) suffered significant str...

Correctness:  2/5
  Reasoning:  The generated answer incorrectly states the total claimed amount as CHF 50,000 instead of the correct CHF 45,000. While it provides detailed breakdowns of damages, the overall total is significantly off, indicating a major error.

Faithfulness: unfaithful
  Unsupported: ['The total claimed amount in this insurance matter is CHF 50,000.', 'Vehicle A (the VW Golf) suffered significant structural damage requiring complete replacement of the front assembly, suspension components, and electrical systems, totalling CHF 18,000.', 'Medical expenses amount to CHF 10,000, covering emergency treatment at Stadtspital Zürich Triemli, specialist consultations.']
  Reasoning:  The generated answer incorrectly

**What to look for:**
- The total is wrong (CHF 50,000 vs CHF 45,000), and the vehicle repair sub-total is also wrong (CHF 18,000 vs CHF 15,000).
- Does the correctness evaluator catch these errors despite the confident, fluent writing?
- Does the faithfulness evaluator flag the wrong amounts as unsupported?

*GPT-4o-mini is usually good enough to catch clear numeric errors. But try the same experiment with more subtle factual mistakes, the results get more interesting.*


### Experiment 3: Terse but Correct

We inject a minimal answer to Q2 (inability to work): just two words: **"6 weeks."** This is technically correct but provides no context, no dates, no citations.

Compare the score here to Experiment 2 (verbose but wrong). What does this tell us about the judge's scoring behaviour?


In [ ]:
TERSE_CORRECT_ANSWER = "6 weeks."

q2 = EVAL_SET[1]["question"]
ref2 = EVAL_SET[1]["reference_answer"]
context2 = MEDICAL_REPORT  # Medical report is most relevant for Q2

print("=== Experiment 3: Terse but Correct ===\n")
print(f"Injected answer: '{TERSE_CORRECT_ANSWER}'\n")

corr_result = evaluate_correctness(q2, ref2, TERSE_CORRECT_ANSWER)
faith_result = evaluate_faithfulness(q2, context2, TERSE_CORRECT_ANSWER)

print(f"Correctness:  {corr_result['score']}/5")
print(f"  Reasoning:  {corr_result['reasoning']}\n")
print(f"Faithfulness: {faith_result['verdict']}")
print(f"  Reasoning:  {faith_result.get('faithfulness_reasoning', faith_result.get('reasoning', ''))}")

print()
print(f"Exp 3 (terse+correct, Q2):       '{TERSE_CORRECT_ANSWER}' — score = {corr_result['score']}/5")


=== Experiment 3: Terse but Correct ===

Injected answer: '6 weeks.'

Correctness:  4/5
  Reasoning:  The generated answer correctly states the duration of time the claimant was unable to work, but it omits the specific dates mentioned in the reference answer, which provides additional context.

Faithfulness: faithful
  Reasoning:  The generated answer of '6 weeks' is directly supported by the context, which states that the patient is deemed fully unfit for work for a period of 6 weeks from the date of the accident.

Exp 3 (terse+correct, Q2):       '6 weeks.' — score = 4/5


**Key question: Is the terse correct answer scored higher or lower than the verbose wrong one?**
- The terse answer is *correct* but omits dates and context. Does the judge penalise for brevity?
- Compare the score here to the verbose-but-wrong answer in Experiment 2.
- If the verbose wrong answer scores higher than the terse correct answer, this is **verbosity bias**.

This is why automated evaluation requires stress-testing. There may be biases in the LLM-judge.


---
## Section 7: Wrap-Up

### Key Takeaways

1. **Evaluating GenAI applications requires fundamentally different approaches than traditional ML.**
   There is no single correct answer, no accuracy metric, and exact string matching is useless.

2. **LLM-as-a-judge is a scalable and practical approach, but it has pitfalls.**
   Position bias, verbosity bias, self-preference and non-determinism are documented failure modes. Always stress-test your evaluators.

3. **Multiple evaluator dimensions give a more complete picture than any single metric.**
   Correctness and faithfulness measure different things and can disagree. Consider running additional evaluators like groundedness, relevance, tone and recall.

4. **Always stress-test your evaluators.**
   The judge can be fooled, especially by confident, fluent text that contains subtle errors.

5. **Human evaluation remains the gold standard.**
   LLM-as-a-judge is a complement, not a replacement. Use it to triage at scale, then review flagged cases with domain experts.

---

### Optional Extensions

If you want to continue after the workshop:

- **Groundedness evaluator:** Require the answer to cite specific passages from the retrieved documents, not just avoid fabrication.
- **Multi-model comparison:** Run the same eval set with `anthropic/claude-haiku-4-5` or `google/gemini-flash-1.5` as the judge model. Do scores agree?
- **RAGAS framework:** Try [github.com/explodinggradients/ragas](https://github.com/explodinggradients/ragas) as an alternative to custom evaluators.
- **Evaluation pipeline:** Wire your evaluators into a CI step that runs on every prompt or retrieval change.
- **Relevance evaluator:** Score whether the retrieved context was actually useful for answering the question.
